![Piggy bank](piggy_bank.jpg)

Personal loans are a lucrative revenue stream for banks. The typical interest rate of a two-year loan in the United Kingdom is [around 10%](https://www.experian.com/blogs/ask-experian/whats-a-good-interest-rate-for-a-personal-loan/). This might not sound like a lot, but in September 2022 alone UK consumers borrowed [around £1.5 billion](https://www.ukfinance.org.uk/system/files/2022-12/Household%20Finance%20Review%202022%20Q3-%20Final.pdf), which would mean approximately £300 million in interest generated by banks over two years!

You have been asked to work with a bank to clean the data they collected as part of a recent marketing campaign, which aimed to get customers to take out a personal loan. They plan to conduct more marketing campaigns going forward so would like you to ensure it conforms to the specific structure and data types that they specify so that they can then use the cleaned data you provide to set up a PostgreSQL database, which will store this campaign's data and allow data from future campaigns to be easily imported. 

They have supplied you with a csv file called `"bank_marketing.csv"`, which you will need to clean, reformat, and split the data, saving three final csv files. Specifically, the three files should have the names and contents as outlined below:

## `client.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `age` | `integer` | Client's age in years | N/A |
| `job` | `object` | Client's type of job | Change `"."` to `"_"` |
| `marital` | `object` | Client's marital status | N/A |
| `education` | `object` | Client's level of education | Change `"."` to `"_"` and `"unknown"` to `np.NaN` |
| `credit_default` | `bool` | Whether the client's credit is in default | Convert to `boolean` data type:<br> `1` if `"yes"`, otherwise `0` |
| `mortgage` | `bool` | Whether the client has an existing mortgage (housing loan) | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0` |

<br>

## `campaign.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `number_contacts` | `integer` | Number of contact attempts to the client in the current campaign | N/A |
| `contact_duration` | `integer` | Last contact duration in seconds | N/A |
| `previous_campaign_contacts` | `integer` | Number of contact attempts to the client in the previous campaign | N/A |
| `previous_outcome` | `bool` | Outcome of the previous campaign | Convert to boolean data type:<br> `1` if `"success"`, otherwise `0`. |
| `campaign_outcome` | `bool` | Outcome of the current campaign | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0`. |
| `last_contact_date` | `datetime` | Last date the client was contacted | Create from a combination of `day`, `month`, and a newly created `year` column (which should have a value of `2022`); <br> **Format =** `"YYYY-MM-DD"` |

<br>

## `economics.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `cons_price_idx` | `float` | Consumer price index (monthly indicator) | N/A |
| `euribor_three_months` | `float` | Euro Interbank Offered Rate (euribor) three-month rate (daily indicator) | N/A |

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("bank_marketing.csv")

df.head()

,client_id,age,job,marital,education,credit_default,mortgage,month,day,contact_duration,number_contacts,previous_campaign_contacts,previous_outcome,cons_price_idx,euribor_three_months,campaign_outcome
0,0,56,housemaid,married,basic.4y,no,no,may,13,261,1,0,nonexistent,93.994,4.857,no
1,1,57,services,married,high.school,unknown,no,may,19,149,1,0,nonexistent,93.994,4.857,no
2,2,37,services,married,high.school,no,yes,may,23,226,1,0,nonexistent,93.994,4.857,no
3,3,40,admin.,married,basic.6y,no,no,may,27,151,1,0,nonexistent,93.994,4.857,no
4,4,56,services,married,high.school,no,no,may,3,307,1,0,nonexistent,93.994,4.857,no


In [6]:
client = df[
    [
        "client_id",
        "age",
        "job",
        "marital",
        "education",
        "credit_default",
        "mortgage"
    ]
].copy()

campaign = df[
    [
        "client_id",
        "number_contacts",
        "contact_duration",
        "previous_campaign_contacts",
        "previous_outcome",
        "campaign_outcome",
        "month",
        "day"
    ]
].copy()

economics = df[
    [
        "client_id",
        "cons_price_idx",
        "euribor_three_months"
    ]
].copy()

print("Client shape:", client.shape)
print("Campaign shape:", campaign.shape)
print("Economics shape:", economics.shape)

Client shape: (41188, 7)
Campaign shape: (41188, 8)
Economics shape: (41188, 3)


In [9]:
print("Before:")
print(client["job"].unique())

client["job"] = client["job"].str.replace(".", "_", regex=False)

print("\nAfter:")
print(client["job"].unique())

Before:
['housemaid' 'services' 'admin_' 'blue-collar' 'technician' 'retired'
 'management' 'unemployed' 'self-employed' 'unknown' 'entrepreneur'
 'student']

After:
['housemaid' 'services' 'admin_' 'blue-collar' 'technician' 'retired'
 'management' 'unemployed' 'self-employed' 'unknown' 'entrepreneur'
 'student']


In [10]:
print("Before:")
print(client["education"].unique())

client["education"] = client["education"].str.replace(
    ".",
    "_",
    regex=False
)

client["education"] = client["education"].replace(
    "unknown",
    np.nan
)

print("\nAfter:")
print(client["education"].unique())

print("\nMissing values:")
print(client["education"].isna().sum())

Before:
['basic.4y' 'high.school' 'basic.6y' 'basic.9y' 'professional.course'
 'unknown' 'university.degree' 'illiterate']

After:
['basic_4y' 'high_school' 'basic_6y' 'basic_9y' 'professional_course' nan
 'university_degree' 'illiterate']

Missing values:
1731


In [11]:
print("Before:")
print(client["credit_default"].value_counts())
print(client["mortgage"].value_counts())

client["credit_default"] = client["credit_default"].eq("yes")
client["mortgage"] = client["mortgage"].eq("yes")

print("\nAfter:")
print(client["credit_default"].value_counts())
print(client["mortgage"].value_counts())

print("\nData types:")
print(client[["credit_default", "mortgage"]].dtypes)

Before:
no         32588
unknown     8597
yes            3
Name: credit_default, dtype: int64
yes        21576
no         18622
unknown      990
Name: mortgage, dtype: int64

After:
False    41185
True         3
Name: credit_default, dtype: int64
True     21576
False    19612
Name: mortgage, dtype: int64

Data types:
credit_default    bool
mortgage          bool
dtype: object


In [12]:
print("Before:")
print(campaign["previous_outcome"].value_counts())
print(campaign["campaign_outcome"].value_counts())

campaign["previous_outcome"] = (
    campaign["previous_outcome"].eq("success")
)

campaign["campaign_outcome"] = (
    campaign["campaign_outcome"].eq("yes")
)

print("\nAfter:")
print(campaign["previous_outcome"].value_counts())
print(campaign["campaign_outcome"].value_counts())

print("\nData types:")
print(
    campaign[
        ["previous_outcome", "campaign_outcome"]
    ].dtypes
)

Before:
nonexistent    35563
failure         4252
success         1373
Name: previous_outcome, dtype: int64
no     36548
yes     4640
Name: campaign_outcome, dtype: int64

After:
False    39815
True      1373
Name: previous_outcome, dtype: int64
False    36548
True      4640
Name: campaign_outcome, dtype: int64

Data types:
previous_outcome    bool
campaign_outcome    bool
dtype: object


In [13]:
campaign["year"] = 2022

campaign["last_contact_date"] = pd.to_datetime(
    campaign["year"].astype(str)
    + "-"
    + campaign["month"]
    + "-"
    + campaign["day"].astype(str)
)

print(
    campaign[
        ["year", "month", "day", "last_contact_date"]
    ].head()
)

print("\nData type:")
print(campaign["last_contact_date"].dtype)

   year month  day last_contact_date
0  2022   may   13        2022-05-13
1  2022   may   19        2022-05-19
2  2022   may   23        2022-05-23
3  2022   may   27        2022-05-27
4  2022   may    3        2022-05-03

Data type:
datetime64[ns]


In [14]:
campaign = campaign.drop(
    columns=["year", "month", "day"]
)

print(campaign.head())

print("\nColumns:")
print(campaign.columns.tolist())

print("\nShape:")
print(campaign.shape)

   client_id  number_contacts  ...  campaign_outcome  last_contact_date
0          0                1  ...             False         2022-05-13
1          1                1  ...             False         2022-05-19
2          2                1  ...             False         2022-05-23
3          3                1  ...             False         2022-05-27
4          4                1  ...             False         2022-05-03

[5 rows x 7 columns]

Columns:
['client_id', 'number_contacts', 'contact_duration', 'previous_campaign_contacts', 'previous_outcome', 'campaign_outcome', 'last_contact_date']

Shape:
(41188, 7)


In [15]:
print(economics.head())

print("\nData types:")
print(economics.dtypes)

print("\nMissing values:")
print(economics.isna().sum())

print("\nShape:")
print(economics.shape)

   client_id  cons_price_idx  euribor_three_months
0          0          93.994                 4.857
1          1          93.994                 4.857
2          2          93.994                 4.857
3          3          93.994                 4.857
4          4          93.994                 4.857

Data types:
client_id                 int64
cons_price_idx          float64
euribor_three_months    float64
dtype: object

Missing values:
client_id               0
cons_price_idx          0
euribor_three_months    0
dtype: int64

Shape:
(41188, 3)


In [16]:
print("CLIENT DATA TYPES")
print(client.dtypes)

print("\nCAMPAIGN DATA TYPES")
print(campaign.dtypes)

print("\nECONOMICS DATA TYPES")
print(economics.dtypes)

print("\nROW COUNTS")
print("client:", len(client))
print("campaign:", len(campaign))
print("economics:", len(economics))

print("\nUNIQUE CLIENT IDs")
print("client:", client["client_id"].nunique())
print("campaign:", campaign["client_id"].nunique())
print("economics:", economics["client_id"].nunique())

print("\nDUPLICATE CLIENT IDs")
print("client:", client["client_id"].duplicated().sum())
print("campaign:", campaign["client_id"].duplicated().sum())
print("economics:", economics["client_id"].duplicated().sum())

print("\nIDS ALIGNED:")
print(
    client["client_id"].equals(campaign["client_id"])
    and client["client_id"].equals(economics["client_id"])
)

CLIENT DATA TYPES
client_id          int64
age                int64
job               object
marital           object
education         object
credit_default      bool
mortgage            bool
dtype: object

CAMPAIGN DATA TYPES
client_id                              int64
number_contacts                        int64
contact_duration                       int64
previous_campaign_contacts             int64
previous_outcome                        bool
campaign_outcome                        bool
last_contact_date             datetime64[ns]
dtype: object

ECONOMICS DATA TYPES
client_id                 int64
cons_price_idx          float64
euribor_three_months    float64
dtype: object

ROW COUNTS
client: 41188
campaign: 41188
economics: 41188

UNIQUE CLIENT IDs
client: 41188
campaign: 41188
economics: 41188

DUPLICATE CLIENT IDs
client: 0
campaign: 0
economics: 0

IDS ALIGNED:
True


In [17]:
client.to_csv(
    "client.csv",
    index=False
)

campaign.to_csv(
    "campaign.csv",
    index=False,
    date_format="%Y-%m-%d"
)

economics.to_csv(
    "economics.csv",
    index=False
)

print("client.csv saved")
print("campaign.csv saved")
print("economics.csv saved")

client.csv saved
campaign.csv saved
economics.csv saved


In [18]:
client_check = pd.read_csv("client.csv")
campaign_check = pd.read_csv("campaign.csv")
economics_check = pd.read_csv("economics.csv")

print("Shapes:")
print("client:", client_check.shape)
print("campaign:", campaign_check.shape)
print("economics:", economics_check.shape)

print("\nClient columns:")
print(client_check.columns.tolist())

print("\nCampaign columns:")
print(campaign_check.columns.tolist())

print("\nEconomics columns:")
print(economics_check.columns.tolist())

print("\nDate sample:")
print(campaign_check["last_contact_date"].head())

print("\nUnexpected index columns:")
print([
    column
    for frame in [client_check, campaign_check, economics_check]
    for column in frame.columns
    if column.startswith("Unnamed")
])

Shapes:
client: (41188, 7)
campaign: (41188, 7)
economics: (41188, 3)

Client columns:
['client_id', 'age', 'job', 'marital', 'education', 'credit_default', 'mortgage']

Campaign columns:
['client_id', 'number_contacts', 'contact_duration', 'previous_campaign_contacts', 'previous_outcome', 'campaign_outcome', 'last_contact_date']

Economics columns:
['client_id', 'cons_price_idx', 'euribor_three_months']

Date sample:
0    2022-05-13
1    2022-05-19
2    2022-05-23
3    2022-05-27
4    2022-05-03
Name: last_contact_date, dtype: object

Unexpected index columns:
[]
